In [1]:
!pip install gradio joblib matplotlib

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
import gradio as gr

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [3]:
from google.colab import files

uploaded = files.upload()

Saving Android_Malware.csv to Android_Malware.csv


In [4]:
import os

print(os.listdir())

['.config', 'Android_Malware.csv', 'sample_data']


In [6]:
df = pd.read_csv("Android_Malware.csv")

print("Dataset Shape :", df.shape)
df.head()

Dataset Shape : (29332, 87)


,android.permission.GET_ACCOUNTS,com.sonyericsson.home.permission.BROADCAST_BADGE,android.permission.READ_PROFILE,android.permission.MANAGE_ACCOUNTS,android.permission.WRITE_SYNC_SETTINGS,android.permission.READ_EXTERNAL_STORAGE,android.permission.RECEIVE_SMS,com.android.launcher.permission.READ_SETTINGS,android.permission.WRITE_SETTINGS,com.google.android.providers.gsf.permission.READ_GSERVICES,...,com.android.launcher.permission.UNINSTALL_SHORTCUT,com.sec.android.iap.permission.BILLING,com.htc.launcher.permission.UPDATE_SHORTCUT,com.sec.android.provider.badge.permission.WRITE,android.permission.ACCESS_NETWORK_STATE,com.google.android.finsky.permission.BIND_GET_INSTALL_REFERRER_SERVICE,com.huawei.android.launcher.permission.READ_SETTINGS,android.permission.READ_SMS,android.permission.PROCESS_INCOMING_CALLS,Result
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,1,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,1,1,0,0,0,0


In [7]:
print(df.info())
print("\nMissing Values:\n")
print(df.isnull().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 29332 entries, 0 to 29331
Data columns (total 87 columns):
 #   Column                                                                         Non-Null Count  Dtype
---  ------                                                                         --------------  -----
 0   android.permission.GET_ACCOUNTS                                                29332 non-null  int64
 1   com.sonyericsson.home.permission.BROADCAST_BADGE                               29332 non-null  int64
 2   android.permission.READ_PROFILE                                                29332 non-null  int64
 3   android.permission.MANAGE_ACCOUNTS                                             29332 non-null  int64
 4   android.permission.WRITE_SYNC_SETTINGS                                         29332 non-null  int64
 5   android.permission.READ_EXTERNAL_STORAGE                                       29332 non-null  int64
 6   android.permission.RECEIVE_SMS        

In [8]:
X = df.drop("Result", axis=1)
y = df["Result"]

print("Features Shape :", X.shape)
print("Target Shape :", y.shape)

Features Shape : (29332, 86)
Target Shape : (29332,)


In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training Features :", X_train.shape)
print("Testing Features :", X_test.shape)

Training Features : (23465, 86)
Testing Features : (5867, 86)


In [11]:
model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

model.fit(X_train, y_train)

print("✅ Model Training Completed")

✅ Model Training Completed


In [12]:
y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)

print("Accuracy :", accuracy)

Accuracy : 0.9708539287540481


In [13]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.97      0.98      0.97      2944
           1       0.98      0.97      0.97      2923

    accuracy                           0.97      5867
   macro avg       0.97      0.97      0.97      5867
weighted avg       0.97      0.97      0.97      5867



In [14]:
joblib.dump(model, "android_malware_detector.pkl")

print("✅ Model Saved Successfully")

✅ Model Saved Successfully


In [15]:
import os

print(os.listdir())

['.config', 'android_malware_detector.pkl', 'Android_Malware.csv', 'sample_data']


In [16]:
def predict_malware(file):

    data = pd.read_csv(file.name)

    if "Result" in data.columns:
        data = data.drop("Result", axis=1)

    prediction = model.predict(data)
    probability = model.predict_proba(data)

    malware = int(sum(prediction))
    safe = len(prediction) - malware

    confidence = np.mean(np.max(probability, axis=1)) * 100

    result = f"""
🛡 AI BASED ANDROID MALWARE DETECTION SYSTEM

━━━━━━━━━━━━━━━━━━━━━━━━━━

📱 Total Apps Scanned : {len(prediction)}

✅ Safe Apps : {safe}

❌ Malware Apps : {malware}

🎯 Average Confidence : {confidence:.2f}%

🤖 Model Accuracy : {accuracy*100:.2f}%

━━━━━━━━━━━━━━━━━━━━━━━━━━

Scan Completed Successfully
"""

    return result

In [17]:
app = gr.Interface(
    fn=predict_malware,
    inputs=gr.File(label="📂 Upload Android Malware CSV"),
    outputs=gr.Textbox(label="📋 Scan Report", lines=15),
    title="🛡 AI Based Android Malware Detection System",
    description="Upload an Android malware dataset (.csv) to detect malicious applications.",
    theme="soft"
)

/usr/local/lib/python3.12/dist-packages/gradio/interface.py:171: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  super().__init__(


In [19]:
import matplotlib.pyplot as plt

def predict_malware(file):

    data = pd.read_csv(file.name)

    if "Result" in data.columns:
        data = data.drop("Result", axis=1)

    prediction = model.predict(data)
    probability = model.predict_proba(data)

    malware = int(sum(prediction))
    safe = len(prediction) - malware

    confidence = np.mean(np.max(probability, axis=1)) * 100

    # Create Pie Chart
    plt.figure(figsize=(5,5))
    plt.pie(
        [safe, malware],
        labels=["Safe", "Malware"],
        autopct="%1.1f%%",
        startangle=90
    )
    plt.title("Malware Detection Result")
    plt.savefig("result_chart.png")
    plt.close()

    result = f"""
🛡 AI BASED ANDROID MALWARE DETECTION SYSTEM

📱 Total Apps Scanned : {len(prediction)}

✅ Safe Apps : {safe}

❌ Malware Apps : {malware}

🎯 Average Confidence : {confidence:.2f}%

🤖 Model Accuracy : {accuracy*100:.2f}%
"""

    return result, "result_chart.png"

In [20]:
app = gr.Interface(
    fn=predict_malware,
    inputs=gr.File(label="📂 Upload Android Malware CSV"),
    outputs=[
        gr.Textbox(label="📋 Scan Report"),
        gr.Image(label="📊 Malware Detection Chart")
    ],
    title="🛡 AI Based Android Malware Detection System",
    description="Upload an Android malware CSV file to detect malware.",
    theme="soft"
)

/usr/local/lib/python3.12/dist-packages/gradio/interface.py:171: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  super().__init__(


In [18]:
app.launch(debug=True)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://972f3ea5be8442c6e3.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


/usr/local/lib/python3.12/dist-packages/gradio/routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://972f3ea5be8442c6e3.gradio.live


In [ ]:
app.launch(debug=True)


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://9631cc6726924306d1.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


/usr/local/lib/python3.12/dist-packages/gradio/routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
